In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# save_perm_pop_TS_all_rdm_combos.py

import os
import numpy as np

from pathlib import Path
from scipy.linalg import toeplitz
from scipy.spatial.distance import pdist
from scipy.stats import rankdata

from google.colab import drive


# ============================================================
# MOUNT DRIVE
# ============================================================

drive.mount("/content/drive", force_remount=False)


# ============================================================
# CONFIGURATION
# ============================================================

SIM_FOLDER = Path(
    "/content/drive/MyDrive/V2_drift/"
    "texturemodel/drift_results/population_simulations"
)

OUT_FOLDER = Path(
    "/content/drive/MyDrive/V2_drift/"
    "texturemodel/drift_results/population_permutations"
)

OUT_FOLDER.mkdir(parents=True, exist_ok=True)


SUBJECTS = list(range(1, 9))
ROIS = ["V1", "V2", "V3", "hV4"]

PCA_TYPE = "pcaHL"
PCA_PCT_VAR = 95
NORM = "raw"

NIMGS = 100
NSESSIONS = 30

NPERMS = 1000
FIXED_FIRST = False
OVERWRITE = True


# Same four RDM analyses as SP
RDM_COMBOS = [
    ("correlation", "spearman"),
    ("correlation", "rmse"),
    ("euclidean", "spearman"),
    ("euclidean", "rmse"),
]


# ============================================================
# PATH HELPERS
# ============================================================

def _sim_path(subject, roi):

    return SIM_FOLDER / (
        f"sim_population_TS_"
        f"{PCA_TYPE}{PCA_PCT_VAR}_{NORM}_"
        f"S{subject:02d}_{roi}_"
        f"Nimg{NIMGS}.npz"
    )


def _metric_tag(rdm_build_metric, rdm_compare_metric):

    build_tag = {
        "correlation": "rdmCorrDist",
        "euclidean": "rdmEuclidean",
    }[rdm_build_metric]

    compare_tag = {
        "spearman": "cmpSpearman",
        "rmse": "cmpRMSE",
    }[rdm_compare_metric]

    return f"{build_tag}_{compare_tag}"


def _out_path(
    subject,
    roi,
    rdm_build_metric,
    rdm_compare_metric,
):

    metric_tag = _metric_tag(
        rdm_build_metric,
        rdm_compare_metric
    )

    fixed = "_fixedFirst" if FIXED_FIRST else ""

    return OUT_FOLDER / (
        f"permPop_TS_"
        f"{PCA_TYPE}{PCA_PCT_VAR}_{NORM}_"
        f"N{NPERMS}{fixed}_"
        f"S{subject:02d}_{roi}_"
        f"{metric_tag}.npz"
    )


# ============================================================
# METRIC HELPERS
# ============================================================

def _nanmean(x, axis=None):
    return np.nanmean(x, axis=axis)


def _pearson_masked(x, y):

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 2:
        return np.nan

    x = x[mask]
    y = y[mask]

    x = x - x.mean()
    y = y - y.mean()

    denom = np.linalg.norm(x) * np.linalg.norm(y)

    if denom == 0:
        return np.nan

    return float((x @ y) / denom)


def _spearman_masked(x, y):

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 2:
        return np.nan

    xr = rankdata(
        x[mask],
        method="average"
    )

    yr = rankdata(
        y[mask],
        method="average"
    )

    return _pearson_masked(xr, yr)


def _rmse_masked(x, y):

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() == 0:
        return np.nan

    diff = x[mask] - y[mask]

    return float(
        np.sqrt(np.mean(diff ** 2))
    )


# ============================================================
# COMPARE SESSION-SPECIFIC RDMS
# ============================================================

def compare_session_rdms(
    sessVec,
    compare_metric
):

    """
    sessVec:
        [session, RDM entries]

    Returns:
        [session, session]

    Spearman:
        higher = more similar geometry

    RMSE:
        higher = more different geometry
    """

    ns = sessVec.shape[0]

    R = np.full(
        (ns, ns),
        np.nan,
        dtype=float
    )

    for i in range(ns):

        if compare_metric == "spearman":
            R[i, i] = 1.0

        elif compare_metric == "rmse":
            R[i, i] = 0.0

        else:
            raise ValueError(
                f"Unknown compare metric: "
                f"{compare_metric}"
            )

        xi = np.asarray(
            sessVec[i],
            dtype=float
        )

        for j in range(i + 1, ns):

            xj = np.asarray(
                sessVec[j],
                dtype=float
            )

            if compare_metric == "spearman":

                val = _spearman_masked(
                    xi,
                    xj
                )

            elif compare_metric == "rmse":

                val = _rmse_masked(
                    xi,
                    xj
                )

            R[i, j] = val
            R[j, i] = val

    return R


# ============================================================
# MAIN ANALYSIS
# ============================================================

def save_perm_pop_TS(
    subject,
    roi,
    rdm_build_metric,
    rdm_compare_metric,
):

    print(
        "\n"
        "============================================================"
    )

    print(
        f"S{subject:02d} {roi} | "
        f"build={rdm_build_metric} | "
        f"compare={rdm_compare_metric}"
    )

    print(
        "============================================================"
    )


    # --------------------------------------------------------
    # Load simulated TS population responses
    # --------------------------------------------------------

    sim_fn = _sim_path(
        subject,
        roi
    )

    if not sim_fn.exists():

        raise FileNotFoundError(
            f"Simulation file not found:\n"
            f"{sim_fn}"
        )


    S = np.load(
        sim_fn,
        allow_pickle=True
    )


    X = np.asarray(
        S["voxSessResp"],
        dtype=float
    )

    # X:
    # voxel × session × image


    nvox, ns, nimg = X.shape


    # --------------------------------------------------------
    # Validate dimensions
    # --------------------------------------------------------

    if ns != NSESSIONS:

        raise RuntimeError(
            f"Expected {NSESSIONS} sessions, "
            f"found {ns}"
        )


    if nimg != NIMGS:

        raise RuntimeError(
            f"Expected {NIMGS} images, "
            f"found {nimg}"
        )


    if not np.all(np.isfinite(X)):

        raise RuntimeError(
            "Non-finite simulated responses found."
        )


    print(
        f"Loaded: vox={nvox}, "
        f"sessions={ns}, "
        f"images={nimg}"
    )


    minSessions = ns


    # ========================================================
    # 1. POPULATION RESPONSE SIMILARITY
    # ========================================================
    #
    # For each image:
    #
    # session × voxel
    #
    # Correlate population vectors between every pair
    # of sessions.
    #
    # Then average over images.
    # ========================================================

    imgCorr = np.full(
        (nimg, ns, ns),
        np.nan,
        dtype=float
    )


    for ii in range(nimg):

        # session × voxel
        M = X[:, :, ii].T

        imgCorr[ii] = np.corrcoef(
            M,
            rowvar=True
        )


    avgImgCorrMat = _nanmean(
        imgCorr,
        axis=0
    )


    # ========================================================
    # 2. RDM CONSTRUCTION
    # ========================================================
    #
    # For each session:
    #
    # image × voxel
    #
    # Construct pairwise distances between images.
    # ========================================================

    nvec = nimg * (nimg - 1) // 2


    sessVec = np.full(
        (ns, nvec),
        np.nan,
        dtype=float
    )


    for ss in range(ns):

        # image × voxel
        Ms = X[:, ss, :].T


        # correlation:
        #     1 - Pearson correlation
        #
        # euclidean:
        #     Euclidean distance

        sessVec[ss] = pdist(
            Ms,
            metric=rdm_build_metric
        )


    # ========================================================
    # 3. COMPARE RDMS ACROSS SESSIONS
    # ========================================================

    R = compare_session_rdms(
        sessVec,
        compare_metric=rdm_compare_metric
    )


    # ========================================================
    # 4. COLLAPSE BY SESSION LAG
    # ========================================================

    distMatrix = toeplitz(
        np.arange(minSessions)
    )


    betweenSessImg = np.full(
        minSessions - 1,
        np.nan
    )


    betweenSessDist = np.full(
        minSessions - 1,
        np.nan
    )


    for d in range(
        1,
        minSessions
    ):

        mask = distMatrix == d


        # Population similarity
        betweenSessImg[d - 1] = _nanmean(
            avgImgCorrMat[mask]
        )


        # RDM similarity/distance
        betweenSessDist[d - 1] = _nanmean(
            R[mask]
        )


    # ========================================================
    # 5. SESSION-ORDER PERMUTATIONS
    # ========================================================

    rng = np.random.default_rng(1)


    if FIXED_FIRST:

        permOrders = np.array(
            [
                np.concatenate(
                    (
                        [0],
                        rng.permutation(
                            minSessions - 1
                        ) + 1
                    )
                )

                for _ in range(NPERMS)
            ],
            dtype=int
        )

    else:

        permOrders = np.array(
            [
                rng.permutation(
                    minSessions
                )

                for _ in range(NPERMS)
            ],
            dtype=int
        )


    # --------------------------------------------------------
    # Allocate permutation results
    # --------------------------------------------------------

    imgCorrMatPerm = np.full(
        (
            NPERMS,
            minSessions,
            minSessions
        ),
        np.nan
    )


    betweenSessImgPerm = np.full(
        (
            NPERMS,
            minSessions - 1
        ),
        np.nan
    )


    betweenSessCorrPerm = np.full(
        (
            NPERMS,
            minSessions,
            minSessions
        ),
        np.nan
    )


    betweenSessDistPerm = np.full(
        (
            NPERMS,
            minSessions - 1
        ),
        np.nan
    )


    # --------------------------------------------------------
    # Permutations
    # --------------------------------------------------------

    for ip, ordv in enumerate(
        permOrders
    ):

        # Population response matrix
        tmp_img = avgImgCorrMat[
            np.ix_(ordv, ordv)
        ]


        imgCorrMatPerm[ip] = tmp_img


        # RDM comparison matrix
        tmp_rdm = R[
            np.ix_(ordv, ordv)
        ]


        betweenSessCorrPerm[ip] = tmp_rdm


        # Collapse permuted matrices by lag
        for d in range(
            1,
            minSessions
        ):

            mask = distMatrix == d


            betweenSessImgPerm[
                ip,
                d - 1
            ] = _nanmean(
                tmp_img[mask]
            )


            betweenSessDistPerm[
                ip,
                d - 1
            ] = _nanmean(
                tmp_rdm[mask]
            )


    # ========================================================
    # 6. SAVE
    # ========================================================

    out_fn = _out_path(
        subject,
        roi,
        rdm_build_metric,
        rdm_compare_metric
    )


    np.savez_compressed(

        out_fn,

        # ---------------- meta ----------------

        subject=np.int32(subject),

        roi=np.array(roi),

        model=np.array("TS"),

        pcaType=np.array(PCA_TYPE),

        pcaPctVar=np.int32(
            PCA_PCT_VAR
        ),

        normalization=np.array(NORM),

        nperms=np.int32(NPERMS),

        fixedFirst=np.bool_(FIXED_FIRST),

        nvox=np.int32(nvox),

        nimages=np.int32(nimg),

        minSessions=np.int32(
            minSessions
        ),

        rdmBuildMetric=np.array(
            rdm_build_metric
        ),

        rdmCompareMetric=np.array(
            rdm_compare_metric
        ),

        metricTag=np.array(
            _metric_tag(
                rdm_build_metric,
                rdm_compare_metric
            )
        ),


        # ---------------- lag ----------------

        distMatrix=distMatrix.astype(
            np.int32
        ),


        # ---------------- population ----------------

        avgImgCorrMat=avgImgCorrMat,

        betweenSessImg=betweenSessImg,


        # ---------------- RDM ----------------

        betweenSessCorr=R,

        betweenSessDist=betweenSessDist,


        # ---------------- permutations ----------------

        permOrders=permOrders,

        imgCorrMatPerm=imgCorrMatPerm,

        betweenSessImgPerm=betweenSessImgPerm,

        betweenSessCorrPerm=betweenSessCorrPerm,

        betweenSessDistPerm=betweenSessDistPerm,
    )


    print(
        f"Saved:\n{out_fn}"
    )


    print(
        f"Population lag values: "
        f"{betweenSessImg.shape}"
    )

    print(
        f"RDM lag values: "
        f"{betweenSessDist.shape}"
    )


    return out_fn


# ============================================================
# BATCH RUNNER
# ============================================================

def run_all_TS():

    all_outputs = []


    for (
        rdm_build_metric,
        rdm_compare_metric
    ) in RDM_COMBOS:

        for roi in ROIS:

            for subject in SUBJECTS:

                sim_fn = _sim_path(
                    subject,
                    roi
                )


                if not sim_fn.exists():

                    print(
                        f"[SKIP] Missing simulation: "
                        f"S{subject:02d} {roi}"
                    )

                    continue


                out_fn = _out_path(
                    subject,
                    roi,
                    rdm_build_metric,
                    rdm_compare_metric
                )


                if (
                    out_fn.exists()
                    and not OVERWRITE
                ):

                    print(
                        f"[SKIP] Exists: "
                        f"S{subject:02d} {roi}"
                    )

                    continue


                try:

                    p = save_perm_pop_TS(
                        subject=subject,
                        roi=roi,
                        rdm_build_metric=rdm_build_metric,
                        rdm_compare_metric=rdm_compare_metric,
                    )

                    all_outputs.append(p)


                except Exception as e:

                    print(
                        f"[ERROR] "
                        f"S{subject:02d} {roi} | "
                        f"{rdm_build_metric}/"
                        f"{rdm_compare_metric}: "
                        f"{e}"
                    )


    print(
        "\n============================================================"
    )

    print("DONE")

    print(
        f"Created/updated "
        f"{len(all_outputs)} files."
    )

    return all_outputs


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    run_all_TS()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

S01 V1 | build=correlation | compare=spearman
Loaded: vox=1350, sessions=30, images=100
Saved:
/content/drive/MyDrive/V2_drift/texturemodel/drift_results/population_permutations/permPop_TS_pcaHL95_raw_N1000_S01_V1_rdmCorrDist_cmpSpearman.npz
Population lag values: (29,)
RDM lag values: (29,)

S02 V1 | build=correlation | compare=spearman
Loaded: vox=1102, sessions=30, images=100
Saved:
/content/drive/MyDrive/V2_drift/texturemodel/drift_results/population_permutations/permPop_TS_pcaHL95_raw_N1000_S02_V1_rdmCorrDist_cmpSpearman.npz
Population lag values: (29,)
RDM lag values: (29,)

S03 V1 | build=correlation | compare=spearman
Loaded: vox=1254, sessions=30, images=100
Saved:
/content/drive/MyDrive/V2_drift/texturemodel/drift_results/population_permutations/permPop_TS_pcaHL95_raw_N1000_S03_V1_rdmCorrDist_cmpSpearman.npz
Population lag values: (29,)
RDM lag val